# 🚀 HUẤN LUYỆN MiT-B0-Lite-KD TỐI ƯU RIÊNG CHO MOBILE FULL-SCAN (960,000 MẪU — COLAB H100)

> **Cải tiến chuyên biệt cho MiT-B0 Mobile Full-Scan (Không dùng Early Exit):**
> * **Student Siêu nhẹ (`MiT-B0-Lite` 2 luồng `RGB + Forensic [DCT, SRM, ELA]`):** `13.74M` tham số (**giảm 8.96×** so với B2), **`29.07 GFLOPs`** / tile `512×512` (**giảm 13.24×** so với B2 `384.75 GFLOPs` và **7.96×** so với B0 cũ `231.48 GFLOPs`).
> * **`LitePrototypeHead`:** Cắt bỏ nút thắt **83.9% FLOPs** (`194.22 → 5.90 GFLOPs`) bằng `1×1 Fusion` + `Depthwise-Separable 3×3`, kết hợp **CrossEntropy + DiceLoss + Dual EMA Prototypes**.
> * **Cross-Modal Knowledge Distillation:** Chưng cất tri thức trực tiếp từ Teacher **`MiT-B2 SOTA` (`19.91% IoU / 33.20% F1`)** qua Foreground-Weighted KL + Soft Dice KD.
> * **Ngân sách huấn luyện chuẩn bài báo gốc ASC-Former (PR 2025):** **`60,000 iters × batch_size 16 = 960,000 crops`** trên GPU **NVIDIA H100 80GB**.
> * **0% lệch miền Train–Deploy:** Student không phụ thuộc PaddleOCR lúc suy luận → xuất ONNX INT8 chỉ **`30.35 MB`**, độ trễ **`~378 ms/tile`** (~3 giây cho Full-Scan 8 tiles mà không cần Early Exit!).

### 🟢 BƯỚC 1: KẾT NỐI DRIVE, CÀI MÔI TRƯỜNG, GIẢI NÉN DATASET & TÌM TEACHER MiT-B2 (~3 Phút)

In [ ]:
import os, sys, glob, torch
from google.colab import drive

print("🖥️ GPU hiện tại:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   VRAM: {vram_gb:.1f} GB | PyTorch: {torch.__version__}")

# 1. Kết nối Google Drive
drive.mount("/content/drive")

# 2. Clone mã nguồn TMD mới nhất từ GitHub
%cd /content
!rm -rf TMD
!git clone https://github.com/gagaduy/TMD.git

sys.path.insert(0, "/content/TMD/ASCFormer")
%env PYTHONPATH=/content/TMD/ASCFormer

# 3. Cài đặt các thư viện cần thiết
!pip install -q yapf==0.40.1 mmcv-lite==2.0.0 mmengine==0.7.0 timm einops prettytable rich onnx onnxruntime

# 4. Giải nén RealTextMan.zip từ Drive sang SSD cục bộ Colab
os.makedirs("/content/TMD/ASCFormer/data/ttd", exist_ok=True)
drive_zip = "/content/drive/MyDrive/RealTextMan.zip"
if not os.path.exists(drive_zip):
    matches = glob.glob("/content/drive/MyDrive/**/RealTextMan.zip", recursive=True)
    if matches:
        drive_zip = matches[0]
    else:
        raise FileNotFoundError("❌ Không tìm thấy RealTextMan.zip trên Google Drive!")

print(f"📦 Đang giải nén {drive_zip} sang SSD Colab...")
!unzip -q -o "{drive_zip}" -d /content/TMD/ASCFormer/data/ttd/
assert os.path.exists("/content/TMD/ASCFormer/data/ttd/RealTextMan/JPEGImages")

# 5. Tìm checkpoint Teacher MiT-B2 SOTA trên Google Drive
teacher_candidates = [
    "/content/drive/MyDrive/RTM_Full60k_B2_SOTA/best_mIoU_iter_30000.pth",
] + sorted(glob.glob("/content/drive/MyDrive/**/RTM_Full60k_B2_SOTA/*best*.pth", recursive=True)) \
  + sorted(glob.glob("/content/drive/MyDrive/**/best_mIoU_iter_30000.pth", recursive=True))

teacher_ckpt = next((p for p in teacher_candidates if os.path.exists(p)), None)
if teacher_ckpt:
    print(f"🎓 Đã tìm thấy Teacher MiT-B2 SOTA: {teacher_ckpt} ({os.path.getsize(teacher_ckpt)/(1024**2):.1f} MB)")
else:
    print("⚠️ CHƯA tìm thấy best_mIoU_iter_30000.pth trên Drive! Nếu muốn chạy KD từ Teacher B2, hãy upload file best_mIoU_iter_30000.pth lên Drive rồi chạy lại cell này.")

print("🎉 BƯỚC 1 HOÀN TẤT!")


### 🔥 BƯỚC 2: HUẤN LUYỆN MiT-B0-Lite-KD (60,000 ITERS × BATCH 16 = 960,000 MẪU) TRÊN H100
> * Tự động lưu checkpoint & `best_mIoU` mỗi `5,000` iters vào `/content/drive/MyDrive/RTM_B0_Lite_KD_960k_H100`.
> * Nếu Colab hết phiên giữa chừng, chỉ cần chạy lại Cell 1 + Cell 2, script sẽ **tự động `--resume` từ mốc gần nhất**!
> * Có thể theo dõi kết quả Validation Full-Scan (1,203 ảnh `val.txt`, `stride=384`) sau mỗi 5,000 iters (`80k` mẫu).

In [ ]:
import os, sys, glob, torch

%cd /content/TMD
!git pull origin main
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer
if "/content/TMD/ASCFormer" not in sys.path:
    sys.path.insert(0, "/content/TMD/ASCFormer")

# Tải trước pretrained MiT-B0
print("📥 Tải sẵn Pretrained SegFormer MiT-B0...")
_ = torch.hub.load_state_dict_from_url(
    "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth",
    map_location="cpu"
)

# Tự động trỏ biến môi trường RTM_TEACHER_CKPT tới Teacher MiT-B2 SOTA trên Drive
teacher_candidates = [
    "/content/drive/MyDrive/RTM_Full60k_B2_SOTA/best_mIoU_iter_30000.pth",
] + sorted(glob.glob("/content/drive/MyDrive/**/RTM_Full60k_B2_SOTA/*best*.pth", recursive=True)) \
  + sorted(glob.glob("/content/drive/MyDrive/**/best_mIoU_iter_30000.pth", recursive=True))
teacher_ckpt = next((p for p in teacher_candidates if os.path.exists(p)), "")
if teacher_ckpt:
    os.environ["RTM_TEACHER_CKPT"] = teacher_ckpt
    print(f"🎓 Bật Knowledge Distillation từ Teacher: {teacher_ckpt}")
else:
    os.environ["RTM_TEACHER_CKPT"] = ""
    print("⚠️ Không có Teacher trên Drive — chạy train MiT-B0-Lite (CE + Dice + Dual Prototype) độc lập.")

CFG_PATH = "configs/ascformer/ascformer_rtm_b0_lite_kd_960k_h100.py"
DRIVE_WORK_DIR = "/content/drive/MyDrive/RTM_B0_Lite_KD_960k_H100"
os.makedirs(DRIVE_WORK_DIR, exist_ok=True)

# Tự động Resume nếu đã có checkpoint trên Drive
ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/iter_*.pth"), key=os.path.getmtime)
resume_flag = f"--resume {ckpts[-1]}" if ckpts else ""
if resume_flag:
    print(f"🔄 Tiếp tục train từ checkpoint: {ckpts[-1]}")
else:
    print("🚀 Bắt đầu huấn luyện MiT-B0-Lite-KD (0 -> 60,000 iters, batch_size=16, tổng 960,000 mẫu)...")

# Tạo worker launcher sạch (vá yapf + torch.load PyTorch 2.6+ + TF32)
with open("/tmp/run_train_kd.py", "w") as wf:
    wf.write("""import sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/train.py'] + sys.argv[1:]
runpy.run_path('tools/train.py', run_name='__main__')
""")
!python /tmp/run_train_kd.py {CFG_PATH} --work-dir {DRIVE_WORK_DIR} {resume_flag}
print("🎉 HOÀN TẤT HUẤN LUYỆN 960K MẪU! Lưu tại:", DRIVE_WORK_DIR)


### 📊 BƯỚC 3: QUÉT ĐA NGƯỠNG FULL-SCAN TRÊN TOÀN BỘ 1,203 ẢNH VAL (`val.txt`)

In [ ]:
import os, glob
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer
os.environ["RTM_TEACHER_CKPT"] = ""  # Không cần nạp Teacher khi đánh giá Student

CFG_PATH = "configs/ascformer/ascformer_rtm_b0_lite_kd_960k_h100.py"
DRIVE_WORK_DIR = "/content/drive/MyDrive/RTM_B0_Lite_KD_960k_H100"
best_ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/best_mIoU_*.pth"), key=os.path.getmtime)
if not best_ckpts:
    best_ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/iter_*.pth"), key=os.path.getmtime)
best_ckpt = best_ckpts[-1]
out_json = f"{DRIVE_WORK_DIR}/Best_B0_Lite_KD_960k_sweep.json"
print(f"🔬 Đang quét đa ngưỡng Full-Scan cho checkpoint tốt nhất: {best_ckpt}")

!python tools/test_threshold_sweep.py "{CFG_PATH}" "{best_ckpt}" \
    --ann-file val.txt \
    --thresholds 0.05 0.10 0.15 0.20 0.25 0.30 0.35 0.40 0.45 0.50 0.55 0.60 \
    --out-json "{out_json}"
print("✅ Đã lưu kết quả sweep đầy đủ vào:", out_json)


### 📦 BƯỚC 4: XUẤT MODEL ONNX INT8 SIÊU NHẸ (`~30 MB`, `29.07 GFLOPs`) SANG GOOGLE DRIVE

In [ ]:
import os, glob
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer
os.environ["RTM_TEACHER_CKPT"] = ""

CFG_PATH = "configs/ascformer/ascformer_rtm_b0_lite_kd_960k_h100.py"
DRIVE_WORK_DIR = "/content/drive/MyDrive/RTM_B0_Lite_KD_960k_H100"
best_ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/best_mIoU_*.pth"), key=os.path.getmtime)
if not best_ckpts:
    best_ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/iter_*.pth"), key=os.path.getmtime)
best_ckpt = best_ckpts[-1]

out_fp32 = f"{DRIVE_WORK_DIR}/rtm_b0_lite_fp32.onnx"
out_int8 = f"{DRIVE_WORK_DIR}/rtm_b0_lite_int8.onnx"

!python tools/export_ondevice_model.py --config "{CFG_PATH}" --checkpoint "{best_ckpt}" --output "{out_fp32}" --fp32
!python tools/export_ondevice_model.py --config "{CFG_PATH}" --checkpoint "{best_ckpt}" --output "{out_int8}"

for p in (out_fp32, out_int8):
    if os.path.exists(p):
        print(f"✅ {os.path.basename(p)}: {os.path.getsize(p)/(1024**2):.2f} MB -> {p}")
